---
# Document Chunking Engine

Converts processed Markdown files into semantically meaningful chunks for RAG.
- Respects document structure (sections, paragraphs)
- Preserves metadata and context
- Optimizes chunk size for embedding models
- Prepares data for vector database ingestion

In [ ]:
import os
import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import tiktoken
from tqdm import tqdm

@dataclass
class ChunkMetadata:
    document_id: str
    chunk_id: str
    section: Optional[str]
    chunk_index: int
    total_chunks: int
    source_file: str
    paper_title: str
    authors: List[str]
    doi: str
    publication_date: str

class DocumentChunker:
    
    def __init__(self, 
                 chunk_size: int = 500,
                 chunk_overlap: int = 50,
                 min_chunk_size: int = 100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.min_chunk_size = min_chunk_size
        
        try:
            self.tokenizer = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer, can try others too, baad mai lets see
        except:
            self.tokenizer = None
            print("Warning: tiktoken not available, using word count approximation")
    
    def count_tokens(self, text: str) -> int:
        if self.tokenizer:
            return len(self.tokenizer.encode(text))
        else:
            return int(len(text.split()) * 1.33)
    
    def extract_sections(self, markdown_content: str) -> List[Dict]:
        sections = []
        lines = markdown_content.split('\n')
        
        current_section = {
            'title': 'Introduction',
            'level': 1,
            'content': [],
            'start_line': 0
        }
        
        for i, line in enumerate(lines):
            if line.strip().startswith('#'):
                # save previous section
                if current_section['content']:
                    current_section['content'] = '\n'.join(current_section['content']).strip()
                    if current_section['content']:
                        sections.append(current_section.copy())
                
                # start new section
                header_level = len(line) - len(line.lstrip('#'))
                title = line.lstrip('#').strip()
                
                current_section = {
                    'title': title,
                    'level': header_level,
                    'content': [],
                    'start_line': i
                }
            else:
                # add content to current section
                if line.strip():  # skip empty lines
                    current_section['content'].append(line)
        
        # add final section
        if current_section['content']:
            current_section['content'] = '\n'.join(current_section['content']).strip()
            if current_section['content']:
                sections.append(current_section)
        
        return sections
    
    def filter_quality_chunks(self, chunks: List[Dict]) -> List[Dict]:
        """Remove low-quality chunks to improve RAG performance"""
        quality_chunks = []
        
        # Sections to skip (metadata, not research content)
        skip_sections = ['viewpoints', 'papers', 'affiliations', 'correspondence', 
                        'figures and tables', 'funding', 'conflicts', 'ethics']
        
        for chunk in chunks:
            content = chunk['content'].lower()
            section = chunk['metadata']['section'].lower()
            
            # Skip metadata sections
            if any(skip in section for skip in skip_sections):
                continue
            
            # Skip very short chunks (likely incomplete)
            if len(chunk['content'].split()) < 25:
                continue
            
            # Skip reference-heavy chunks
            if (content.count('doi:') > 3 or 
                content.count('et al') > 6 or
                content.count('pmid:') > 3 or
                content.count('http') > 4):
                continue
            
            # Skip chunks that are mostly author names/numbers
            words = chunk['content'].split()
            if (len(words) < 40 and 
                (sum(1 for w in words if w.isdigit()) > len(words) * 0.3 or
                 sum(1 for w in words if w[0].isupper() and len(w) > 2) > len(words) * 0.5)):
                continue
            
            quality_chunks.append(chunk)
        
        return quality_chunks
    
    def process_document(self, md_file_path: str, metadata_file_path: str = None) -> List[Dict]:
        
        with open(md_file_path, 'r', encoding='utf-8') as f:
            markdown_content = f.read()
        
        document_metadata = {'document_id': Path(md_file_path).stem, 'source_file': md_file_path}
        if metadata_file_path and os.path.exists(metadata_file_path):
            with open(metadata_file_path, 'r', encoding='utf-8') as f:
                loaded_metadata = json.load(f)
                document_metadata.update(loaded_metadata)
        
        sections = self.extract_sections(markdown_content)
        
        all_chunks = []
        global_chunk_index = 0 
        
        for section_idx, section in enumerate(sections):
            section_chunks = self.chunk_section(section, document_metadata, global_chunk_index, section_idx)
            all_chunks.extend(section_chunks)
            global_chunk_index += len(section_chunks)  
        
        for chunk in all_chunks:
            chunk['metadata']['total_chunks'] = len(all_chunks)
        
        return all_chunks

    def chunk_section(self, section: Dict, document_metadata: Dict, start_chunk_index: int = 0, section_idx: int = 0) -> List[Dict]:
        """Chunk a single section into optimal sizes"""
        content = section['content']
        section_title = section['title']
        
        if not content.strip():
            return []
        
        chunks = []
        
        paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]
        
        current_chunk = ""
        chunk_index = start_chunk_index 
        
        for paragraph in paragraphs:

            potential_chunk = current_chunk + "\n\n" + paragraph if current_chunk else paragraph
            token_count = self.count_tokens(potential_chunk)
            
            if token_count <= self.chunk_size:
                current_chunk = potential_chunk
            else:

                if current_chunk and self.count_tokens(current_chunk) >= self.min_chunk_size:
                    chunks.append(self._create_chunk(
                        current_chunk, 
                        chunk_index, 
                        section_title, 
                        document_metadata,
                        section_idx
                    ))
                    chunk_index += 1
                
                if self.count_tokens(paragraph) <= self.chunk_size:
                    current_chunk = paragraph
                else:

                    sentence_chunks = self._split_long_paragraph(paragraph, section_title, document_metadata, chunk_index, section_idx)
                    chunks.extend(sentence_chunks)
                    chunk_index += len(sentence_chunks)
                    current_chunk = ""
        
        if current_chunk and self.count_tokens(current_chunk) >= self.min_chunk_size:
            chunks.append(self._create_chunk(
                current_chunk, 
                chunk_index, 
                section_title, 
                document_metadata,
                section_idx
            ))
        
        return chunks

    def _create_chunk(self, content: str, chunk_index: int, section_title: str, document_metadata: Dict, section_idx: int = 0) -> Dict:

        # use both section and chunk index for unique IDs
        chunk_id = f"{document_metadata['document_id']}_sec{section_idx:02d}_chunk{chunk_index:03d}"
        
        return {
            'chunk_id': chunk_id,
            'content': content.strip(),
            'token_count': self.count_tokens(content),
            'metadata': {
                'document_id': document_metadata['document_id'],
                'section': section_title,
                'section_index': section_idx,
                'chunk_index': chunk_index,
                'source_file': document_metadata['source_file'],
                'paper_title': document_metadata.get('title', 'Unknown'),
                'authors': document_metadata.get('authors', []),
                'doi': document_metadata.get('doi', ''),
                'publication_date': document_metadata.get('publication_date', ''),
                'source_url': document_metadata.get('source_url', '')
            }
        }

    def _split_long_paragraph(self, paragraph: str, section_title: str, document_metadata: Dict, start_index: int, section_idx: int = 0) -> List[Dict]:
        """Split overly long paragraphs by sentences"""
        sentences = re.split(r'(?<=[.!?])\s+', paragraph)
        chunks = []
        current_chunk = ""
        chunk_index = start_index
        
        for sentence in sentences:
            potential_chunk = current_chunk + " " + sentence if current_chunk else sentence
            
            if self.count_tokens(potential_chunk) <= self.chunk_size:
                current_chunk = potential_chunk
            else:
                if current_chunk:
                    chunks.append(self._create_chunk(
                        current_chunk, 
                        chunk_index, 
                        section_title, 
                        document_metadata,
                        section_idx
                    ))
                    chunk_index += 1
                
                current_chunk = sentence
        
        if current_chunk:
            chunks.append(self._create_chunk(
                current_chunk, 
                chunk_index, 
                section_title, 
                document_metadata,
                section_idx
            ))
        
        return chunks
    
    def process_directory(self, input_dir: str, output_dir: str, max_files: int = None) -> Dict:

        input_path = Path(input_dir)
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        md_files = list(input_path.glob("*.md"))
        if max_files:
            md_files = md_files[:max_files]
        
        all_chunks = []
        processing_stats = {
            'total_documents': len(md_files),
            'total_chunks': 0,
            'filtered_chunks': 0,
            'avg_tokens_per_chunk': 0,
            'sections_processed': 0
        }
        
        print(f"Processing {len(md_files)} markdown files...")
        
        for md_file in tqdm(md_files, desc="Chunking documents"):
            # Look for metadata file
            metadata_file = md_file.with_name(f"{md_file.stem}_processed_meta.json")
            metadata_path = metadata_file if metadata_file.exists() else None
            
            # Process document
            document_chunks = self.process_document(str(md_file), str(metadata_path) if metadata_path else None)
            all_chunks.extend(document_chunks)
            
            print(f"  {md_file.name}: {len(document_chunks)} chunks")
        
        print("Filtering low-quality chunks...")
        quality_chunks = self.filter_quality_chunks(all_chunks)
        
        chunks_file = output_path / "all_chunks.json"
        with open(chunks_file, 'w', encoding='utf-8') as f:
            json.dump(quality_chunks, f, indent=2, ensure_ascii=False)
        
        doc_index = {}
        for chunk in quality_chunks:
            doc_id = chunk['metadata']['document_id']
            if doc_id not in doc_index:
                doc_index[doc_id] = {
                    'chunk_count': 0,
                    'sections': set(),
                    'chunk_ids': []
                }
            doc_index[doc_id]['chunk_count'] += 1
            doc_index[doc_id]['sections'].add(chunk['metadata']['section'])
            doc_index[doc_id]['chunk_ids'].append(chunk['chunk_id'])
        
        for doc_id in doc_index:
            doc_index[doc_id]['sections'] = list(doc_index[doc_id]['sections'])
        
        index_file = output_path / "document_index.json"
        with open(index_file, 'w', encoding='utf-8') as f:
            json.dump(doc_index, f, indent=2)
        
        processing_stats['total_chunks'] = len(all_chunks)
        processing_stats['filtered_chunks'] = len(quality_chunks)
        if quality_chunks:
            processing_stats['avg_tokens_per_chunk'] = sum(c['token_count'] for c in quality_chunks) / len(quality_chunks)
        
        stats_file = output_path / "chunking_stats.json"
        with open(stats_file, 'w', encoding='utf-8') as f:
            json.dump(processing_stats, f, indent=2)
        
        print(f"\n{'='*60}")
        print("CHUNKING COMPLETE")
        print(f"{'='*60}")
        print(f"Total documents: {processing_stats['total_documents']}")
        print(f"Raw chunks: {processing_stats['total_chunks']}")
        print(f"Quality chunks: {processing_stats['filtered_chunks']}")
        print(f"Filtering efficiency: {(processing_stats['filtered_chunks']/processing_stats['total_chunks']*100):.1f}%")
        print(f"Average tokens per chunk: {processing_stats['avg_tokens_per_chunk']:.1f}")
        print(f"Output saved to: {chunks_file}")
        print(f"Document index: {index_file}")
        
        return processing_stats

# Usage
if __name__ == "__main__":
    chunker = DocumentChunker(
        chunk_size=500,  # Target chunk size in tokens
        chunk_overlap=50,  # Overlap between chunks
        min_chunk_size=100  # Minimum chunk size
    )
    
    input_directory = "D:/PsyWiz/processed_md"
    output_directory = "D:/PsyWiz/chunks"
    
    # For full processing, remove max_files limit
    stats = chunker.process_directory(
        input_directory, 
        output_directory, 
        max_files=None  # Process all files
    )

Processing 2 markdown files...


Chunking documents:  50%|█████     | 1/2 [00:00<00:00,  9.09it/s]

  article_2.md: 36 chunks


Chunking documents: 100%|██████████| 2/2 [00:00<00:00,  7.48it/s]

  article_3.md: 68 chunks
Filtering low-quality chunks...

CHUNKING COMPLETE
Total documents: 2
Raw chunks: 104
Quality chunks: 87
Filtering efficiency: 83.7%
Average tokens per chunk: 305.0
Output saved to: D:\PsyWiz\chunks\all_chunks.json
Document index: D:\PsyWiz\chunks\document_index.json


---

New SPlit Engine

In [2]:
"""
Enhanced Document Chunking Engine for PsyWiz

Improvements:
- Proper metadata extraction from JSON files
- Better content extraction from structured MD files
- Enhanced chunk quality filtering
- Richer metadata preservation
"""

import os
import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import tiktoken
from tqdm import tqdm

@dataclass
class ChunkMetadata:
    document_id: str
    chunk_id: str
    section: Optional[str]
    chunk_index: int
    total_chunks: int
    source_file: str
    paper_title: str
    authors: List[str]
    doi: str
    publication_date: str
    journal: str
    source_url: str
    content_preview: str

class EnhancedDocumentChunker:
    
    def __init__(self, 
                 chunk_size: int = 500,
                 chunk_overlap: int = 50,
                 min_chunk_size: int = 100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.min_chunk_size = min_chunk_size
        
        try:
            self.tokenizer = tiktoken.get_encoding("cl100k_base")
        except:
            self.tokenizer = None
            print("Warning: tiktoken not available, using word count approximation")
    
    def count_tokens(self, text: str) -> int:
        if self.tokenizer:
            return len(self.tokenizer.encode(text))
        else:
            return int(len(text.split()) * 1.33)
    
    def load_metadata(self, metadata_file_path: str) -> Dict:
        """Load and normalize metadata from JSON file."""
        try:
            with open(metadata_file_path, 'r', encoding='utf-8') as f:
                raw_metadata = json.load(f)
            
            # Normalize metadata fields to handle different naming conventions
            normalized_metadata = {
                'title': self._extract_title(raw_metadata),
                'authors': self._extract_authors(raw_metadata),
                'doi': raw_metadata.get('citation_doi', raw_metadata.get('doi', '')),
                'publication_date': self._extract_date(raw_metadata),
                'journal': self._extract_journal(raw_metadata),
                'source_url': raw_metadata.get('source_url', ''),
                'abstract': raw_metadata.get('abstract', ''),
                'keywords': raw_metadata.get('keywords', [])
            }
            
            return normalized_metadata
            
        except Exception as e:
            print(f"Warning: Could not load metadata from {metadata_file_path}: {str(e)}")
            return self._get_default_metadata()
    
    def _extract_title(self, metadata: Dict) -> str:
        """Extract title from various possible fields."""
        title_fields = ['citation_title', 'title', 'title_from_results', 'paper_title']
        for field in title_fields:
            if metadata.get(field):
                return metadata[field].strip()
        return "Unknown Title"
    
    def _extract_authors(self, metadata: Dict) -> List[str]:
        """Extract authors from metadata."""
        # Handle different author formats
        if 'citation_author' in metadata:
            authors_str = metadata['citation_author']
            if isinstance(authors_str, str):
                # Split by common separators
                authors = re.split(r',\s*|\s+and\s+|\s*;\s*', authors_str)
                return [author.strip() for author in authors if author.strip()]
            elif isinstance(authors_str, list):
                return authors_str
        
        if 'authors' in metadata:
            if isinstance(metadata['authors'], list):
                return metadata['authors']
            elif isinstance(metadata['authors'], str):
                return [metadata['authors']]
        
        return []
    
    def _extract_date(self, metadata: Dict) -> str:
        """Extract publication date."""
        date_fields = ['citation_publication_date', 'publication_date', 'date', 'year']
        for field in date_fields:
            if metadata.get(field):
                return str(metadata[field]).strip()
        return ""
    
    def _extract_journal(self, metadata: Dict) -> str:
        """Extract journal name."""
        journal_fields = ['citation_journal_title', 'journal', 'journal_title', 'venue']
        for field in journal_fields:
            if metadata.get(field):
                return metadata[field].strip()
        return ""
    
    def _get_default_metadata(self) -> Dict:
        """Return default metadata when JSON loading fails."""
        return {
            'title': 'Unknown Title',
            'authors': [],
            'doi': '',
            'publication_date': '',
            'journal': '',
            'source_url': '',
            'abstract': '',
            'keywords': []
        }
    
    def extract_sections(self, markdown_content: str) -> List[Dict]:
        """Extract sections from improved markdown structure."""
        sections = []
        lines = markdown_content.split('\n')
        
        current_section = {
            'title': 'Document Header',
            'level': 1,
            'content': [],
            'start_line': 0
        }
        
        # Skip metadata lines at the top (HTML comments, etc.)
        content_start = 0
        for i, line in enumerate(lines):
            if line.strip() and not line.strip().startswith('<!--'):
                content_start = i
                break
        
        for i, line in enumerate(lines[content_start:], content_start):
            line = line.strip()
            
            # Detect headers (##, ###, etc.)
            if line.startswith('#') and not line.startswith('<!-- '):
                # Save previous section
                if current_section['content']:
                    content = '\n'.join(current_section['content']).strip()
                    if content and len(content.split()) > 10:  # Only keep substantial sections
                        current_section['content'] = content
                        sections.append(current_section.copy())
                
                # Start new section
                header_level = len(line) - len(line.lstrip('#'))
                title = line.lstrip('#').strip()
                
                current_section = {
                    'title': title if title else 'Untitled Section',
                    'level': header_level,
                    'content': [],
                    'start_line': i
                }
            else:
                # Add content to current section (skip empty lines and HTML comments)
                if line and not line.startswith('<!--') and not line.startswith('<'):
                    current_section['content'].append(line)
        
        # Add final section
        if current_section['content']:
            content = '\n'.join(current_section['content']).strip()
            if content and len(content.split()) > 10:
                current_section['content'] = content
                sections.append(current_section)
        
        return sections
    
    def filter_quality_chunks(self, chunks: List[Dict]) -> List[Dict]:
        """Enhanced quality filtering for research papers."""
        quality_chunks = []
        
        # Sections to skip (metadata, not research content)
        skip_sections = [
            'viewpoints', 'affiliations', 'correspondence', 'funding', 
            'conflicts', 'ethics', 'copyright', 'acknowledgements',
            'author contributions', 'declaration of interest', 'references',
            'data availability', 'supplementary', 'icmje forms'
        ]
        
        # Sections to prioritize (high-value research content)
        priority_sections = [
            'abstract', 'introduction', 'background', 'method', 'methodology',
            'results', 'findings', 'discussion', 'conclusion', 'implications',
            'analysis', 'study design', 'participants', 'procedure'
        ]
        
        for chunk in chunks:
            content = chunk['content'].lower()
            section = chunk['metadata']['section'].lower()
            
            # Skip metadata sections
            if any(skip in section for skip in skip_sections):
                continue
            
            # Skip very short chunks
            if len(chunk['content'].split()) < 25:
                continue
            
            # Skip reference-heavy chunks
            if (content.count('doi:') > 3 or 
                content.count('et al') > 6 or
                content.count('pmid:') > 3 or
                content.count('http') > 4 or
                content.count('fig.') > 3 or
                content.count('table') > 3):
                continue
            
            # Skip chunks that are mostly numbers/citations
            words = chunk['content'].split()
            if (len(words) < 40 and 
                (sum(1 for w in words if w.isdigit()) > len(words) * 0.3 or
                 sum(1 for w in words if re.match(r'^\([0-9]+\)$', w)) > len(words) * 0.2)):
                continue
            
            # Boost priority section chunks
            if any(priority in section for priority in priority_sections):
                chunk['metadata']['priority_score'] = 1.0
            else:
                chunk['metadata']['priority_score'] = 0.7
            
            quality_chunks.append(chunk)
        
        return quality_chunks
    
    def process_document(self, md_file_path: str, metadata_file_path: str = None) -> List[Dict]:
        """Process a single document with enhanced metadata extraction."""
        
        # Load markdown content
        with open(md_file_path, 'r', encoding='utf-8') as f:
            markdown_content = f.read()
        
        # Initialize document metadata
        document_id = Path(md_file_path).stem
        document_metadata = {
            'document_id': document_id,
            'source_file': md_file_path
        }
        
        # Load rich metadata from JSON file
        if metadata_file_path and os.path.exists(metadata_file_path):
            rich_metadata = self.load_metadata(metadata_file_path)
            document_metadata.update(rich_metadata)
        else:
            # Try to find metadata file by convention
            potential_meta_file = Path(md_file_path).with_name(f"{document_id}_meta.json")
            if potential_meta_file.exists():
                rich_metadata = self.load_metadata(str(potential_meta_file))
                document_metadata.update(rich_metadata)
            else:
                print(f"Warning: No metadata file found for {md_file_path}")
                document_metadata.update(self._get_default_metadata())
        
        # Extract sections
        sections = self.extract_sections(markdown_content)
        
        if not sections:
            print(f"Warning: No sections found in {md_file_path}")
            return []
        
        # Process sections into chunks
        all_chunks = []
        global_chunk_index = 0
        
        for section_idx, section in enumerate(sections):
            section_chunks = self.chunk_section(
                section, document_metadata, global_chunk_index, section_idx
            )
            all_chunks.extend(section_chunks)
            global_chunk_index += len(section_chunks)
        
        # Update total chunks count
        for chunk in all_chunks:
            chunk['metadata']['total_chunks'] = len(all_chunks)
        
        return all_chunks
    
    def chunk_section(self, section: Dict, document_metadata: Dict, start_chunk_index: int = 0, section_idx: int = 0) -> List[Dict]:
        """Enhanced section chunking with better content preservation."""
        content = section['content']
        section_title = section['title']
        
        if not content.strip():
            return []
        
        chunks = []
        paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]
        
        current_chunk = ""
        chunk_index = start_chunk_index
        
        for paragraph in paragraphs:
            # Clean paragraph content
            paragraph = self._clean_paragraph(paragraph)
            if not paragraph:
                continue
            
            potential_chunk = current_chunk + "\n\n" + paragraph if current_chunk else paragraph
            token_count = self.count_tokens(potential_chunk)
            
            if token_count <= self.chunk_size:
                current_chunk = potential_chunk
            else:
                # Save current chunk if it meets minimum size
                if current_chunk and self.count_tokens(current_chunk) >= self.min_chunk_size:
                    chunks.append(self._create_enhanced_chunk(
                        current_chunk, chunk_index, section_title, 
                        document_metadata, section_idx
                    ))
                    chunk_index += 1
                
                # Handle oversized paragraphs
                if self.count_tokens(paragraph) <= self.chunk_size:
                    current_chunk = paragraph
                else:
                    sentence_chunks = self._split_long_paragraph(
                        paragraph, section_title, document_metadata, chunk_index, section_idx
                    )
                    chunks.extend(sentence_chunks)
                    chunk_index += len(sentence_chunks)
                    current_chunk = ""
        
        # Save final chunk
        if current_chunk and self.count_tokens(current_chunk) >= self.min_chunk_size:
            chunks.append(self._create_enhanced_chunk(
                current_chunk, chunk_index, section_title, 
                document_metadata, section_idx
            ))
        
        return chunks
    
    def _clean_paragraph(self, paragraph: str) -> str:
        """Clean paragraph content for better quality."""
        # Remove excessive whitespace
        paragraph = re.sub(r'\s+', ' ', paragraph).strip()
        
        # Remove HTML tags and comments
        paragraph = re.sub(r'<[^>]+>', '', paragraph)
        paragraph = re.sub(r'<!--.*?-->', '', paragraph)
        
        # Remove table markers and excessive punctuation
        paragraph = re.sub(r'\|+', '', paragraph)
        paragraph = re.sub(r'-{3,}', '', paragraph)
        
        # Skip very short or low-content paragraphs
        if len(paragraph.split()) < 5:
            return ""
        
        return paragraph
    
    def _create_enhanced_chunk(self, content: str, chunk_index: int, section_title: str, 
                             document_metadata: Dict, section_idx: int = 0) -> Dict:
        """Create chunk with enhanced metadata."""
        
        chunk_id = f"{document_metadata['document_id']}_sec{section_idx:02d}_chunk{chunk_index:03d}"
        
        # Create content preview (first 100 characters)
        content_preview = content[:100] + "..." if len(content) > 100 else content
        
        return {
            'chunk_id': chunk_id,
            'content': content.strip(),
            'token_count': self.count_tokens(content),
            'metadata': {
                # Document identifiers
                'document_id': document_metadata['document_id'],
                'chunk_id': chunk_id,
                'source_file': document_metadata['source_file'],
                
                # Paper metadata (from JSON)
                'paper_title': document_metadata.get('title', 'Unknown Title'),
                'authors': document_metadata.get('authors', []),
                'doi': document_metadata.get('doi', ''),
                'publication_date': document_metadata.get('publication_date', ''),
                'journal': document_metadata.get('journal', ''),
                'source_url': document_metadata.get('source_url', ''),
                
                # Section information
                'section': section_title,
                'section_index': section_idx,
                'chunk_index': chunk_index,
                
                # Content metadata
                'content_preview': content_preview,
                'token_count': self.count_tokens(content),
                
                # Additional metadata
                'abstract': document_metadata.get('abstract', '')[:200] + "..." if document_metadata.get('abstract', '') else '',
                'keywords': document_metadata.get('keywords', [])
            }
        }
    
    def _split_long_paragraph(self, paragraph: str, section_title: str, 
                            document_metadata: Dict, start_index: int, section_idx: int = 0) -> List[Dict]:
        """Split overly long paragraphs by sentences."""
        sentences = re.split(r'(?<=[.!?])\s+', paragraph)
        chunks = []
        current_chunk = ""
        chunk_index = start_index
        
        for sentence in sentences:
            potential_chunk = current_chunk + " " + sentence if current_chunk else sentence
            
            if self.count_tokens(potential_chunk) <= self.chunk_size:
                current_chunk = potential_chunk
            else:
                if current_chunk:
                    chunks.append(self._create_enhanced_chunk(
                        current_chunk, chunk_index, section_title, 
                        document_metadata, section_idx
                    ))
                    chunk_index += 1
                
                current_chunk = sentence
        
        if current_chunk:
            chunks.append(self._create_enhanced_chunk(
                current_chunk, chunk_index, section_title, 
                document_metadata, section_idx
            ))
        
        return chunks
    
    def process_directory(self, input_dir: str, output_dir: str, max_files: int = None) -> Dict:
        """Process directory with enhanced metadata handling."""
        
        input_path = Path(input_dir)
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        # Find all markdown files
        md_files = list(input_path.glob("*.md"))
        if max_files:
            md_files = md_files[:max_files]
        
        all_chunks = []
        processing_stats = {
            'total_documents': len(md_files),
            'successful_documents': 0,
            'failed_documents': 0,
            'total_chunks': 0,
            'filtered_chunks': 0,
            'avg_tokens_per_chunk': 0,
            'metadata_success_rate': 0
        }
        
        successful_metadata = 0
        
        print(f"Processing {len(md_files)} markdown files...")
        
        for md_file in tqdm(md_files, desc="Enhanced chunking"):
            try:
                # Find corresponding metadata file
                metadata_file = md_file.with_name(f"{md_file.stem}_meta.json")
                metadata_path = metadata_file if metadata_file.exists() else None
                
                if metadata_path:
                    successful_metadata += 1
                
                # Process document
                document_chunks = self.process_document(str(md_file), str(metadata_path) if metadata_path else None)
                
                if document_chunks:
                    all_chunks.extend(document_chunks)
                    processing_stats['successful_documents'] += 1
                    print(f"  ✓ {md_file.name}: {len(document_chunks)} chunks")
                else:
                    processing_stats['failed_documents'] += 1
                    print(f"  ✗ {md_file.name}: Failed to process")
                    
            except Exception as e:
                processing_stats['failed_documents'] += 1
                print(f"  ✗ {md_file.name}: Error - {str(e)}")
        
        print("Filtering low-quality chunks...")
        quality_chunks = self.filter_quality_chunks(all_chunks)
        
        # Save processed chunks
        chunks_file = output_path / "all_chunks.json"
        with open(chunks_file, 'w', encoding='utf-8') as f:
            json.dump(quality_chunks, f, indent=2, ensure_ascii=False)
        
        # Create enhanced document index
        doc_index = {}
        metadata_summary = {}
        
        for chunk in quality_chunks:
            doc_id = chunk['metadata']['document_id']
            if doc_id not in doc_index:
                doc_index[doc_id] = {
                    'chunk_count': 0,
                    'sections': set(),
                    'chunk_ids': [],
                    'paper_title': chunk['metadata']['paper_title'],
                    'authors': chunk['metadata']['authors'],
                    'doi': chunk['metadata']['doi'],
                    'journal': chunk['metadata']['journal'],
                    'publication_date': chunk['metadata']['publication_date']
                }
                
                metadata_summary[doc_id] = {
                    'title': chunk['metadata']['paper_title'],
                    'authors': chunk['metadata']['authors'],
                    'doi': chunk['metadata']['doi'],
                    'source_url': chunk['metadata']['source_url']
                }
            
            doc_index[doc_id]['chunk_count'] += 1
            doc_index[doc_id]['sections'].add(chunk['metadata']['section'])
            doc_index[doc_id]['chunk_ids'].append(chunk['chunk_id'])
        
        # Convert sets to lists for JSON serialization
        for doc_id in doc_index:
            doc_index[doc_id]['sections'] = list(doc_index[doc_id]['sections'])
        
        # Save enhanced index
        index_file = output_path / "all_documents_index.json"
        with open(index_file, 'w', encoding='utf-8') as f:
            json.dump(doc_index, f, indent=2, ensure_ascii=False)
        
        # Save metadata summary
        metadata_file = output_path / "document_metadata_summary.json"
        with open(metadata_file, 'w', encoding='utf-8') as f:
            json.dump(metadata_summary, f, indent=2, ensure_ascii=False)
        
        # Calculate statistics
        processing_stats['total_chunks'] = len(all_chunks)
        processing_stats['filtered_chunks'] = len(quality_chunks)
        processing_stats['metadata_success_rate'] = (successful_metadata / len(md_files)) * 100
        
        if quality_chunks:
            processing_stats['avg_tokens_per_chunk'] = sum(c['token_count'] for c in quality_chunks) / len(quality_chunks)
        
        # Save enhanced statistics
        stats_file = output_path / "final_chunking_stats.json"
        with open(stats_file, 'w', encoding='utf-8') as f:
            json.dump(processing_stats, f, indent=2)
        
        # Print enhanced summary
        print(f"\n{'='*70}")
        print("Op CHUNKING COMPLETE")
        print(f"{'='*70}")
        print(f"📊 Documents processed: {processing_stats['successful_documents']}/{processing_stats['total_documents']}")
        print(f"📊 Metadata success rate: {processing_stats['metadata_success_rate']:.1f}%")
        print(f"📊 Raw chunks created: {processing_stats['total_chunks']}")
        print(f"📊 Quality chunks retained: {processing_stats['filtered_chunks']}")
        print(f"📊 Filtering efficiency: {(processing_stats['filtered_chunks']/processing_stats['total_chunks']*100):.1f}%")
        print(f"📊 Average tokens per chunk: {processing_stats['avg_tokens_per_chunk']:.1f}")
        print(f"\n📁 Output files:")
        print(f"   • Chunks: {chunks_file}")
        print(f"   • Index: {index_file}")
        print(f"   • Metadata: {metadata_file}")
        print(f"   • Stats: {stats_file}")
        
        return processing_stats

# Usage with your paths
if __name__ == "__main__":
    chunker = EnhancedDocumentChunker(
        chunk_size=500,  # Target chunk size in tokens
        chunk_overlap=50,  # Overlap between chunks
        min_chunk_size=100  # Minimum chunk size
    )
    
    # Updated paths based on your folder structure
    input_directory = "D:/PsyWiz/raw_data"  # Your raw_data folder with .md and .json files
    output_directory = "D:/PsyWiz/updated_chunks"
    
    # Process all files
    stats = chunker.process_directory(
        input_directory, 
        output_directory, 
        max_files=None  # Process all files, or set to 10 for testing
    )
    
    print(f"\n🎉 Enhanced chunking completed!")
    print(f"Now your RAG system will have rich metadata including:")
    print(f"  • Complete paper titles")
    print(f"  • Author names") 
    print(f"  • DOIs and publication dates")
    print(f"  • Journal names")
    print(f"  • Source URLs")
    print(f"  • Content previews")

Processing 198 markdown files...


Enhanced chunking:   1%|          | 1/198 [00:00<00:29,  6.69it/s]

  ✓ article_1.md: 25 chunks


Enhanced chunking:   1%|          | 2/198 [00:00<00:44,  4.44it/s]

  ✓ article_10.md: 27 chunks


...

Enhanced chunking:  99%|█████████▉| 197/198 [01:05<00:00,  3.12it/s]

  ✓ article_98.md: 52 chunks


Enhanced chunking: 100%|██████████| 198/198 [01:06<00:00,  2.99it/s]

  ✓ article_99.md: 33 chunks
Filtering low-quality chunks...



Op CHUNKING COMPLETE
📊 Documents processed: 197/198
📊 Metadata success rate: 100.0%
📊 Raw chunks created: 6163
📊 Quality chunks retained: 4579
📊 Filtering efficiency: 74.3%
📊 Average tokens per chunk: 323.6

📁 Output files:
   • Chunks: D:\PsyWiz\updated_chunks\all_chunks.json
   • Index: D:\PsyWiz\updated_chunks\all_documents_index.json
   • Metadata: D:\PsyWiz\updated_chunks\document_metadata_summary.json
   • Stats: D:\PsyWiz\updated_chunks\final_chunking_stats.json

🎉 Enhanced chunking completed!
Now your RAG system will have rich metadata including:
  • Complete paper titles
  • Author names
  • DOIs and publication dates
  • Journal names
  • Source URLs
  • Content previews
